In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 7.7 Bose–Einstein and Fermi–Dirac: The Grand Canonical Derivation

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VII — Quantum Statistical Mechanics",
    number="7.7",
    title="Bose–Einstein and Fermi–Dirac: The Grand Canonical Derivation",
    blurb="Three times this volume has met the same two functions — in a contour integral, "
    "in a warm qubit, in a warm oscillator — and here, at last, is the reason. In the "
    "grand canonical ensemble the occupation of every quantum mode becomes an "
    "independent little system: two states for a fermion, a ladder for a boson, and "
    "the bookkeeping falls open in one line each. The distributions of quantum "
    "statistics follow, their fluctuations bunch or hush according to species, and a "
    "quiet convergence condition on the boson side is left glowing — it will "
    "condense a gas before the volume is done.",
    difficulty="advanced",
    estimate="190–230 min",
)

## Notebook overview

Movement II opens with the volume's central derivation. Movement I kept meeting two functions by
accident: the Matsubara contours of [§7.2](complex-analysis-applications.ipynb) produced them from pure complex analysis, the warm qubit of [§7.4](thermal-density-matrix.ipynb)
*was* the Fermi function, the warm oscillator of [§7.5](oscillator-at-temperature.ipynb) *was* the Bose function. This notebook derives
both on purpose, from the grand canonical ensemble of [§5.9](../05-classical-stat-mech/grand-canonical-ensemble-equivalence.ipynb) quantized, and the derivation
explains all three accidents at once.

The technical miracle to watch for is **factorization**. For identical particles ([§6.20](../06-quantum-mechanics/identical-particles.ipynb)) the
many-body basis is labelled by *occupation numbers*: how many particles sit in each
single-particle mode, with $n_i \in \{0, 1\}$ for fermions and $n_i \in \{0, 1, 2, \dots\}$ for
bosons. At fixed total $N$ the constraint $\sum_i n_i = N$ couples every mode to every other, and
the canonical combinatorics is famously miserable. Release $N$ — trade it for a chemical
potential, exactly the move [§5.9](../05-classical-stat-mech/grand-canonical-ensemble-equivalence.ipynb) built — and the grand partition function *factorizes over modes*,
$\Xi = \prod_i \Xi_i$, with each factor trivial: two terms for a fermion mode, a geometric series
for a boson mode. We verify the factorization not by argument but *against the entire Fock
space*, configuration by configuration, to ten digits. One line of algebra per species then
yields the pair

$$
n_F(\varepsilon) = \frac{1}{e^{\beta(\varepsilon-\mu)} + 1},
\qquad
n_B(\varepsilon) = \frac{1}{e^{\beta(\varepsilon-\mu)} - 1},
$$

and Movement I's accidents become theorems: a fermionic mode *is* a two-level system (the qubit,
with $\mu$ the price of occupation), a bosonic mode *is* a harmonic oscillator (the ladder, with
$\mu = 0$ for photons previewed). The boson factor's convergence condition — $\mu < \varepsilon_
{\min}$, a *ceiling* on the chemical potential — is flagged loudly and left glowing: when the
physics later pushes $\mu$ against that ceiling, the ground mode's occupation will blow up, and
that mechanism is Bose–Einstein condensation (the BEC notebook; the bounded $g_{3/2}$ of [§7.3](statistical-toolkit.ipynb) is this
ceiling's integral shadow).

The rest of the notebook is verification with teeth. The **rendezvous** is held numerically:
the ensemble $n_B$ and $n_F$ reproduce the Matsubara closed forms of [§7.2](complex-analysis-applications.ipynb) to seven digits and Movement
I's single-system results identically — three independent routes, one pair of functions, and the
volume's methodological moral made explicit. The **Maxwell–Boltzmann limit** is common to both
statistics, with the telltale ordering $n_B > \text{MB} > n_F$ visible already in the dilute
corrections (where classicality is *valid* is the business of [§7.8](classical-limit-thermal-wavelength.ipynb), deferred). **Ensemble equivalence**
is re-enacted in quantum dress by exact fixed-$N$ enumeration, its finite-size deviations
shrinking on the $1/M$ schedule [§5.9](../05-classical-stat-mech/grand-canonical-ensemble-equivalence.ipynb) promised. And the **fluctuations** unify the movement:
$\langle\Delta n^2\rangle = \langle n\rangle(1 \pm \langle n\rangle)$ — bosons bunch (the
geometric variance of [§7.5](oscillator-at-temperature.ipynb), now general, and explosive near the ceiling), fermions antibunch, their
variance capped at $\tfrac14$: Pauli, audible in the noise.

> **Conventions (this notebook).** Units $k_B = 1$; fugacity $z = e^{\beta\mu}$. The boson
> formulas require $\mu < \varepsilon_{\min}$ and the code *guards* that domain rather than
> trusting the caller. Enumerations state their sizes: fermion Fock spaces via
> `itertools.product([0,1], ...)`, fixed-$N$ canonical states via `itertools.combinations`,
> boson occupation grids via `numpy.meshgrid` with a stated $n_{\max}$ governed by the geometric
> tail rule $n_{\max} \gg 1/(1 - ze^{-\beta\varepsilon})$ — and where a truncation residual
> survives, it is printed, not hidden. Far-forward notebooks are cited *by topic* (the BEC
> notebook, the photon-gas notebook) pending the volume's renumbering; near references ([§7.8](classical-limit-thermal-wavelength.ipynb)) by
> number.
>
> **How to read the checks.** Each exercise closes with a `validate` call against an independent
> fact: the factorized $\Xi$ against the brute Fock-space sum to ten digits (both species); the
> Fermi step sharpening with $n_F(\mu) = \tfrac12$ pinned; the Planck occupation recovered at
> $\mu = 0$ by a genuinely different route (ladder sums, not the same formula); the Matsubara
> rendezvous at seven digits; the MB ordering with its $\pm$MB corrections; canonical-vs-grand
> deviations halving as the system doubles; and the variance formulas $n(1\pm n)$ against brute
> enumeration, with the near-ceiling boson mode at variance $\approx 20.6$ and the fermionic
> $\tfrac14$ cap saturated at half filling. A ✓ is strong evidence; a ✗ is a prompt to *locate
> the discrepancy*.
>
> **Scope.** The two distributions and their fluctuations, derived and verified. The occupation
> *labels* suffice here — the operator machinery that creates and destroys quanta (second
> quantization) is the many-body volume's opening and is named as a horizon. The degeneracy
> criterion and the $N!$ derivation are [§7.8](classical-limit-thermal-wavelength.ipynb); the Fermi step goes to work in the fermion movement;
> the ceiling saturates in the BEC notebook. See Pathria & Beale (Chs. 6–8); Huang (Chs. 8–9);
> Kardar, *Statistical Physics of Particles* (Ch. 7). Cross-reference [§5.9](../05-classical-stat-mech/grand-canonical-ensemble-equivalence.ipynb) (the grand ensemble,
> put to work), [§6.20](../06-quantum-mechanics/identical-particles.ipynb) (the occupation restrictions), [§7.2](complex-analysis-applications.ipynb)/[§7.4](thermal-density-matrix.ipynb)/[§7.5](oscillator-at-temperature.ipynb) (the rendezvous parties), [§7.3](statistical-toolkit.ipynb) (the
> integrals awaiting these occupations).

## Theory in brief

### Occupation numbers: the right coordinates for identical particles

For identical particles the question "which particle is in which state?" is meaningless — [§6.20](../06-quantum-mechanics/identical-particles.ipynb)
made that a postulate — and the surviving good question is "*how many* particles occupy each
single-particle mode?" A many-body basis state is fully specified by the list of **occupation
numbers**,

```{math}
:label: eq-occupation
|\{n_i\}\rangle,
\qquad
n_i \in \begin{cases} \{0, 1\} & \text{fermions (Pauli)}\\ \{0, 1, 2, \dots\} & \text{bosons,}\end{cases}
\qquad
E = \sum_i n_i\varepsilon_i,
\quad
N = \sum_i n_i,
```

with the fermion restriction inherited directly from antisymmetry: a doubly-occupied
antisymmetric state is its own negative, hence zero. One honest sentence of horizon: the operator
machinery that *creates and destroys* quanta (second quantization) belongs to the many-body
volume; for equilibrium statistics the occupation labels suffice, and they are all we use.

### The grand canonical ensemble, quantized

Recall [§5.9](../05-classical-stat-mech/grand-canonical-ensemble-equivalence.ipynb) (invoked, not re-taught): when a system exchanges particles with a reservoir, the
right ensemble weights each state by $z^N e^{-\beta E}$ with fugacity $z = e^{\beta\mu}$, and the
normalization is $\Xi = \sum_N z^N Z_N$. In occupation coordinates the double sum over $N$ and
states collapses into *one unconstrained sum* over all $\{n_i\}$:

```{math}
:label: eq-grand-quantized
\Xi = \sum_{\{n_i\}} \prod_i \left(z\,e^{-\beta\varepsilon_i}\right)^{n_i}
= \prod_i \underbrace{\sum_{n} \left(z\,e^{-\beta\varepsilon_i}\right)^{n}}_{\Xi_i}
= \prod_i \Xi_i .
```

The middle step is the miracle, and it is worth naming what makes it possible. At fixed $N$ the
constraint $\sum_i n_i = N$ couples every mode to every other — the sum does *not* factorize, and
the canonical combinatorics of identical particles is notoriously unpleasant. Releasing $N$
decouples the modes completely: each mode becomes its own small system in contact with the
reservoir, and *this* is why the classical volume built [§5.9](../05-classical-stat-mech/grand-canonical-ensemble-equivalence.ipynb). The promise is kept.

### The Fermi–Dirac distribution

A fermion mode has exactly two Fock states, $n = 0$ and $n = 1$:

```{math}
:label: eq-fermi-dirac
\Xi_i = 1 + z\,e^{-\beta\varepsilon_i},
\qquad
\langle n_i\rangle = z\,\frac{\partial \ln\Xi_i}{\partial z}
= \frac{1}{e^{\beta(\varepsilon_i-\mu)} + 1} \equiv n_F(\varepsilon_i).
```

Limits carry the physics: as $T \to 0$, $n_F$ sharpens into the **step** $\theta(\mu -
\varepsilon)$ — filled below the chemical potential, empty above, with $n_F(\mu) = \tfrac12$
always — and $\mu(T{=}0) = \varepsilon_F$ is the **Fermi energy**, the organizing scale of the
fermion movement to come. The Movement-I accident is now a theorem: a fermion mode *is* the
two-level system of [§7.4](thermal-density-matrix.ipynb), with the occupation costing $\varepsilon - \mu$ rather than $\varepsilon$ —
the identification is exact, not analogical.

### The Bose–Einstein distribution, and the glowing constraint

A boson mode is a geometric series — and a geometric series has a convergence condition:

```{math}
:label: eq-bose-einstein
\Xi_i = \sum_{n=0}^{\infty}\left(z\,e^{-\beta\varepsilon_i}\right)^n
= \frac{1}{1 - z\,e^{-\beta\varepsilon_i}}
\quad\big(\mu < \varepsilon_{\min}\big),
\qquad
\langle n_i\rangle = \frac{1}{e^{\beta(\varepsilon_i-\mu)} - 1} \equiv n_B(\varepsilon_i).
```

Flag the constraint loudly, because the volume will return to it: the chemical potential of
bosons is **capped below the lowest level**. As $\mu$ approaches $\varepsilon_{\min}$ the ground
mode's occupation diverges — and when particle conservation later *pushes* $\mu$ against that
ceiling, the gas will have nowhere to put its particles but the ground mode. That mechanism is
Bose–Einstein condensation (the BEC notebook; the bounded $g_{3/2}$ of [§7.3](statistical-toolkit.ipynb) is the ceiling's integral
shadow). The second Movement-I accident is also now a theorem: a boson mode *is* the harmonic
oscillator of [§7.5](oscillator-at-temperature.ipynb), its quanta the rungs — and photons, whose number nothing conserves, sit at $\mu = 0$
(previewed for the photon-gas notebook).

### The rendezvous: three routes, one pair of functions

The volume now holds three independent derivations of the same pair:

```{math}
:label: eq-rendezvous
\underbrace{\frac{1}{\beta}\sum_n \frac{1}{\omega_n^2+\varepsilon^2} =
\frac{1}{2\varepsilon}\big(1 \pm 2n_{B/F}\big)}_{\text{§7.2: contours}},
\qquad
\underbrace{p_1^{\text{qubit}} = n_F}_{\text{§7.4}},
\qquad
\underbrace{\langle n\rangle^{\text{ladder}} = n_B}_{\text{§7.5}},
```

and Exercise 5 makes them shake hands numerically. A contour integral that knew nothing of
ensembles, two exactly solvable single systems that knew nothing of each other, and a reservoir
argument had no obligation to agree; they agree because each is a window onto the same object —
the thermal state of a mode. That is the volume's methodological moral, made plain.

### The Maxwell–Boltzmann limit, with its ordering

For $z \ll 1$ both distributions forget their statistics:

```{math}
:label: eq-mb-limit
n_{B/F} = \frac{1}{z^{-1}e^{\beta\varepsilon} \mp 1}
\;\xrightarrow{\;z\to0\;}\;
z\,e^{-\beta\varepsilon} \equiv n_{\text{MB}},
\qquad
n_B = n_{\text{MB}}\big(1 + n_{\text{MB}} + \dots\big),
\quad
n_F = n_{\text{MB}}\big(1 - n_{\text{MB}} + \dots\big),
```

the classical Boltzmann occupation. But keep the next order and the species never quite forget:
$n_B > n_{\text{MB}} > n_F$ *always* — bosons over-occupy and fermions under-occupy the classical
expectation even in the dilute limit, the first quantitative glimpse of bunching and exclusion.
*Where* the classical description is valid — the degeneracy criterion $n\lambda^3 \ll 1$ and the
thermal wavelength behind it — is the next notebook's entire business, deferred explicitly.

### Ensemble equivalence, quantum edition

The grand ensemble was a convenience, and physics at fixed $N$ must agree in the large-system
limit — the theme of [§5.9](../05-classical-stat-mech/grand-canonical-ensemble-equivalence.ipynb), now re-enacted with Pauli in the game:

```{math}
:label: eq-bfd-equivalence
\langle n_i\rangle_{\text{canonical}, N}
\;\xrightarrow{\;M \to \infty\;}\;
n_F(\varepsilon_i)\Big|_{\mu:\ \sum_i n_F(\varepsilon_i) = N},
```

verified by *exact enumeration*: every fixed-$N$ fermion configuration via
`itertools.combinations`, against the grand formulas with $\mu$ matched by root finding. The
deviations shrink like $1/M$ as the system doubles. Small systems care which ensemble describes
them; matter does not.

### Fluctuations: bunching and antibunching

One more derivative of $\ln\Xi_i$ unifies the movement's noise story:

```{math}
:label: eq-bfd-fluctuations
\langle\Delta n^2\rangle = \big(z\,\partial_z\big)^2 \ln\Xi_i
= \langle n\rangle\big(1 \pm \langle n\rangle\big)
\qquad(+\ \text{bosons},\ -\ \text{fermions}).
```

The plus sign is bosonic **bunching** — the super-Poissonian geometric variance of [§7.5](oscillator-at-temperature.ipynb), now general,
and explosive near the fugacity ceiling. The minus sign is fermionic **antibunching**: the
variance $n(1-n)$ is capped at $\tfrac14$, saturated at half filling — Pauli suppressing
occupation noise, the quietest statistics the world allows. One formula, one sign, and the
difference between laser physics and electron shot noise (horizons, named).

## Setup

In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import brentq

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# Conventions: k_B = 1; fugacity z = e^(βμ). Boson formulas exist only
# for μ < ε_min and the helpers GUARD that domain (a ValueError beats a silent
# negative occupation). Enumerations state their sizes; boson occupation grids obey
# the geometric-tail rule n_max >> 1/(1 − z e^(−βε)), and surviving truncation
# residuals are printed, never hidden. Far-forward notebooks are cited BY TOPIC
# (the BEC notebook, the photon-gas notebook) pending the volume's renumbering.


def n_FD(eps, beta, mu):
    """The Fermi–Dirac occupation n_F = 1/(e^(β(ε−μ)) + 1) (eq-fermi-dirac).

    The +1 keeps the denominator ≥ 1 for every argument, so no domain guard is needed
    — a fermion mode is defined at any μ, hot or cold, filled or empty. Vectorized
    over ε.

    Parameters
    ----------
    eps : float or numpy.ndarray
        Mode energy (or energies).
    beta : float
        Inverse temperature.
    mu : float
        Chemical potential.

    Returns
    -------
    float or numpy.ndarray
        The mean occupation, in (0, 1).
    """
    return 1.0 / (np.exp(beta * (np.asarray(eps) - mu)) + 1.0)


def n_BE(eps, beta, mu):
    """The Bose–Einstein occupation n_B = 1/(e^(β(ε−μ)) − 1), guarded (eq-bose-einstein).

    The geometric series behind this formula converges only for μ < ε: the helper
    raises rather than returning the negative occupation the bare arithmetic would
    produce past the ceiling. The denominator is numpy.expm1(β(ε−μ)) — the small-x
    discipline of §7.5: near the ceiling β(ε−μ) is small and a naive e^x − 1 loses digits
    exactly where the occupation is largest.

    Parameters
    ----------
    eps : float or numpy.ndarray
        Mode energy (or energies), all strictly greater than mu.
    beta : float
        Inverse temperature.
    mu : float
        Chemical potential, mu < min(eps).

    Returns
    -------
    float or numpy.ndarray
        The mean occupation, unbounded above as ε → μ.
    """
    eps_arr = np.asarray(eps, dtype=float)
    if np.any(eps_arr <= mu):
        raise ValueError("Bose occupation requires mu < eps: the fugacity ceiling")
    return 1.0 / np.expm1(beta * (eps_arr - mu))


def xi_mode(eps, beta, mu, statistics):
    """The single-mode grand partition function Ξ_i, closed form (eq-grand-quantized).

    Fermions: the two-term sum 1 + z e^(−βε) (empty or occupied — Pauli's whole Fock
    space). Bosons: the geometric series 1/(1 − z e^(−βε)), guarded by its convergence
    condition z e^(−βε) < 1, i.e. μ < ε. These closed forms are what Exercise 2 checks
    against brute Fock-space enumeration, configuration by configuration.

    Parameters
    ----------
    eps : float
        Mode energy.
    beta : float
        Inverse temperature.
    mu : float
        Chemical potential.
    statistics : str
        'fermi' or 'bose'.

    Returns
    -------
    float
        Ξ_i for the mode.
    """
    w = np.exp(beta * (mu - eps))  # z e^(−βε)
    if statistics == "fermi":
        return 1.0 + w
    if statistics == "bose":
        if w >= 1.0:
            raise ValueError("boson mode requires z e^(−βε) < 1: the fugacity ceiling")
        return 1.0 / (1.0 - w)
    raise ValueError("statistics must be 'fermi' or 'bose'")


def grand_enumerate(eps_list, beta, mu, statistics, n_max=1):
    """Brute-force grand-canonical sums over the (truncated) Fock space of a few modes.

    The definitional check on factorization: sum the weight z^(Σn) e^(−β Σ n ε) over
    every occupation configuration and read off Ξ, ⟨n_i⟩, and ⟨Δn_i^2⟩ with no closed
    form anywhere. Fermions enumerate {0,1}^M with itertools.product; bosons enumerate
    a numpy.meshgrid grid of occupations 0..n_max per mode, with the truncation
    governed by the geometric tail (z e^(−βε))^(n_max) — choose n_max >> 1/(1 − z
    e^(−βε)) of the SOFTEST mode, and note that second moments need more headroom
    than means (the n^2 weighting fattens the tail).

    Parameters
    ----------
    eps_list : sequence of float
        The mode energies (keep M small: the grid has (n_max+1)^M points).
    beta : float
        Inverse temperature.
    mu : float
        Chemical potential (mu < min(eps) required for bosons).
    statistics : str
        'fermi' or 'bose'.
    n_max : int, optional
        Per-mode occupation cutoff for bosons (ignored for fermions, where it is 1).

    Returns
    -------
    tuple
        (Xi, mean_occupations, variances), the latter two as numpy arrays per mode.
    """
    eps_arr = np.asarray(eps_list, dtype=float)
    M = eps_arr.size
    if statistics == "fermi":
        configs = np.array(list(itertools.product([0, 1], repeat=M)), dtype=float)
        n_grid = [configs[:, i] for i in range(M)]
    elif statistics == "bose":
        occ = [np.arange(n_max + 1, dtype=float)] * M
        mesh = np.meshgrid(*occ, indexing="ij")
        n_grid = [g.ravel() for g in mesh]
    else:
        raise ValueError("statistics must be 'fermi' or 'bose'")
    log_w = sum(n_grid[i] * (beta * (mu - eps_arr[i])) for i in range(M))
    w = np.exp(log_w)
    Xi = float(w.sum())
    means = np.array([float((n_grid[i] * w).sum()) / Xi for i in range(M)])
    second = np.array([float((n_grid[i] ** 2 * w).sum()) / Xi for i in range(M)])
    return Xi, means, second - means**2


def canonical_occupations(eps_list, N, beta):
    """Exact fixed-N fermion occupations by enumerating itertools.combinations.

    The canonical ensemble done with no approximation at all: every N-subset of the M
    modes is one antisymmetric basis state with weight e^(−β Σ ε), and ⟨n_i⟩ is the
    weighted fraction of subsets containing mode i. The cost is combinatorial —
    C(16, 8) = 12870 is comfortable, but the point of Exercise 7 is the TREND toward
    the grand-canonical formulas, not heroic system sizes.

    Parameters
    ----------
    eps_list : sequence of float
        The M mode energies.
    N : int
        The fixed particle number, 0 ≤ N ≤ M.
    beta : float
        Inverse temperature.

    Returns
    -------
    numpy.ndarray
        The exact canonical ⟨n_i⟩, one entry per mode.
    """
    eps_arr = np.asarray(eps_list, dtype=float)
    M = eps_arr.size
    Z = 0.0
    occ = np.zeros(M)
    for filled in itertools.combinations(range(M), N):
        w = float(np.exp(-beta * eps_arr[list(filled)].sum()))
        Z += w
        for i in filled:
            occ[i] += w
    return occ / Z


def mu_from_N(eps_list, beta, N_target):
    """The chemical potential μ with grand-canonical ⟨N⟩ = Σ n_F(ε_i) = N_target, by brentq.

    Σ n_F is strictly increasing in μ (each term is), running from 0 to M across
    μ = −∞..+∞, so a bracketing root find is guaranteed a unique solution for any
    0 < N_target < M. The bracket pads the spectrum by 30/β on each side — enough for
    occupations within e^(−30) of their saturated values. (Fermion side only: no
    ceiling to respect. A boson-side matcher would need its bracket capped below
    ε_min.)

    Parameters
    ----------
    eps_list : sequence of float
        The mode energies.
    beta : float
        Inverse temperature.
    N_target : float
        The desired mean particle number, strictly between 0 and M.

    Returns
    -------
    float
        The matching chemical potential.
    """
    eps_arr = np.asarray(eps_list, dtype=float)

    def excess(mu):
        return float(np.sum(n_FD(eps_arr, beta, mu))) - N_target

    lo = float(eps_arr.min()) - 30.0 / beta
    hi = float(eps_arr.max()) + 30.0 / beta
    return brentq(excess, lo, hi, xtol=1e-13)

## Exercise 1 — Occupation numbers, and the ensemble that sets them free

The right coordinates for identical particles, and why fixed $N$ was the obstacle all along.
Cite {eq}`eq-occupation`, {eq}`eq-grand-quantized`.

1. State the occupation-number description of the many-body basis for identical particles, with
   the Pauli restriction $n_i \in \{0,1\}$ for fermions inherited from the antisymmetry of [§6.20](../06-quantum-mechanics/identical-particles.ipynb).
2. Write $\Xi = \sum_N z^N Z_N$ ([§5.9](../05-classical-stat-mech/grand-canonical-ensemble-equivalence.ipynb), invoked) as one unconstrained sum over $\{n_i\}$ and derive
   the factorization $\Xi = \prod_i \Xi_i$.
3. Explain (prose) what fixed $N$ destroys: the constraint $\sum n_i = N$ couples every mode —
   exhibit the obstruction on a two-mode fermion example by writing out $Z_{N=1}$ explicitly and
   checking the definitional double sum against the factorized $\Xi$ numerically.
4. State the payoff: each mode is now its own small system against a reservoir — the two
   distributions will each cost one line.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    Xi_double_sum,
    Xi_factorized,
    "the grand ensemble decouples the modes: Σ_N z^N Z_N = Π Ξ_i on the worked two-mode example",
    rtol=1e-12,
)

## Exercise 2 — Factorization, brutally verified

The claim $\Xi = \prod\Xi_i$ is checked against the entire Fock space, configuration by
configuration. Cite {eq}`eq-grand-quantized`.

1. For three fermion modes ($\varepsilon = 0.3, 0.9, 1.6$; $\beta = 1.1$; $\mu = 0.1$),
   enumerate all $2^3$ occupation configurations with `itertools.product([0,1], repeat=3)` (the
   `grand_enumerate` helper) and sum the weights $z^{\sum n}e^{-\beta\sum n\varepsilon}$.
2. Confirm the brute $\Xi$ equals $\prod(1 + ze^{-\beta\varepsilon_i})$ to at least ten digits,
   and the brute $\langle n_i\rangle$ equal the $n_F$ formula to at least eight digits.
3. Repeat for bosons on a truncated `numpy.meshgrid` occupation grid ($n_{\max} = 110$ for the
   gates; a deliberately short $n_{\max} = 40$ run demonstrates the tail rule $n_{\max} \gg
   1/(1 - ze^{-\beta\varepsilon})$), confirming the product of geometric series — and *state*
   the residual of the $n_{\max} = 80$ run of record (${\sim}10^{-6}$ on the softest mode's
   occupation).
4. Note (prose) which mode carries the residual and why: $ze^{-\beta\varepsilon} = 0.80$ there —
   the geometric series is slow exactly where the physics is about to become interesting.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    [Xi_brute_F, Xi_brute_B],
    [Xi_product_F, Xi_product_B],
    "Ξ factorizes over modes: verified against the whole Fock space, both statistics",
    rtol=1e-10,
)
validate.close(
    n_brute_F,
    n_formula_F,
    "the brute fermion occupations equal the n_F closed form",
    rtol=1e-10,
)
validate.close(
    n_brute_B,
    n_formula_B,
    "the brute boson occupations equal the n_B closed form (n_max = 110)",
    rtol=1e-8,
)
validate.check(
    resid_40 > 1e-4 and resid_80 < 1e-4,
    "the geometric tail rule: n_max = 40 is visibly short on the r = 0.80 mode, 80 is not",
    f"residuals {resid_40:.1e} → {resid_80:.1e}",
)

## Exercise 3 — The Fermi–Dirac distribution

Two Fock states, one line, and the step that organizes all of metal physics. Cite
{eq}`eq-fermi-dirac`.

1. Derive $n_F = 1/(e^{\beta(\varepsilon-\mu)}+1)$ from $\Xi_i = 1 + ze^{-\beta\varepsilon_i}$
   via $\langle n\rangle = z\,\partial_z\ln\Xi_i$.
2. Verify the $T \to 0$ step numerically ($T = 0.5, 0.1, 0.02$ at $\mu = 0.9$): filled below
   $\mu$, empty above, $n_F(\mu) = \tfrac12$ always.
3. Identify $\mu(T{=}0)$ as the Fermi energy $\varepsilon_F$ (the highest filled level) and
   preview its role as the organizing scale of the fermion movement.
4. Close the Movement-I accident (prose plus a one-line check): the mode *is* the two-level
   system of [§7.4](thermal-density-matrix.ipynb) with the occupation cost $\varepsilon - \mu$; the qubit's upper population is $n_F$
   identically.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    n_at_mu_all,
    [0.5, 0.5, 0.5],
    "n_F(μ) = 1/2 at every temperature: the point the smearing never moves",
    rtol=1e-12,
)
validate.check(
    step_rows[0.02][0] > 0.99 and step_rows[0.02][2] < 0.01,
    "the T → 0 step: filled below μ, empty above, to better than 1% at 5 thermal widths",
    f"n(μ−5T) = {step_rows[0.02][0]:.4f}, n(μ+5T) = {step_rows[0.02][2]:.4f}",
)
validate.check(
    accident_gap < 1e-14,
    "the Movement-I accident closed: the qubit of §7.4 at Δ = ε − μ IS the fermion mode",
    f"max deviation {accident_gap:.1e}",
)

## Exercise 4 — The Bose–Einstein distribution, and the ceiling

A geometric series with a convergence condition that will later condense a gas. Cite
{eq}`eq-bose-einstein`.

1. Derive $n_B = 1/(e^{\beta(\varepsilon-\mu)}-1)$ from the geometric $\Xi_i =
   1/(1-ze^{-\beta\varepsilon_i})$, stating the convergence requirement $\mu < \varepsilon_{\min}$.
2. Demonstrate the approach to the ceiling numerically: fix $\varepsilon_{\min} = 0.5$ and raise
   $\mu$ toward it, tabulating the ground mode's $\langle n\rangle$ and the enumeration
   $n_{\max}$ the tail rule demands — occupation and cost both diverging.
3. Close the second Movement-I accident: the mode *is* the oscillator of [§7.5](oscillator-at-temperature.ipynb); at $\mu = 0$ the
   formula is the Planck occupation, checked by a genuinely different route (a truncated
   Boltzmann ladder sum, not the same closed form).
4. Flag the physics to come (prose): a gas whose $\mu$ is pushed against $\varepsilon_{\min}$ by
   particle conservation has nowhere to put its particles but the ground mode — the mechanism,
   named now, demonstrated in the BEC notebook.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    n_B_mu0,
    planck_ladder,
    "the Bose–Einstein distribution at μ = 0 is the Planck occupation (ladder-sum route)",
    rtol=1e-12,
)
validate.check(
    all(a < b for a, b in zip(ceiling_occ, ceiling_occ[1:])) and ceiling_occ[-1] > 99.0,
    "the ceiling approached: the ground occupation diverges as μ → ε_min",
    f"⟨n⟩ = {ceiling_occ[0]:.2f} → {ceiling_occ[-1]:.1f}",
)
validate.check(
    guard_works,
    "the domain guard: n_BE refuses μ ≥ ε_min rather than returning unphysical occupations",
)

## Exercise 5 — The rendezvous: three routes, one pair of functions

Contours, small systems, and ensembles are made to shake hands, numerically. Cite
{eq}`eq-rendezvous`.

1. Recompute the Matsubara sums of [§7.2](complex-analysis-applications.ipynb) $(1/\beta)\sum_n 1/(\omega_n^2+\varepsilon^2)$ for both
   frequency families (symmetric truncation $n_{\max} = 3\times10^5$ with the stated $1/n_{\max}$
   tail estimate) at $\varepsilon = 1.3$, $\beta = 2$.
2. Confirm they equal $(1/2\varepsilon)(1+2n_B)$ and $(1/2\varepsilon)(1-2n_F)$ with the
   *ensemble* $n$'s of this notebook, to at least six digits.
3. Confirm the qubit population of [§7.4](thermal-density-matrix.ipynb) and the Planck occupation of [§7.5](oscillator-at-temperature.ipynb) equal $n_F$ and $n_B$ identically
   at matched parameters.
4. Reflect (prose): three derivations with no shared machinery agreed because each computes the
   same object — the volume's methodological moral.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    [mats_bose, mats_fermi],
    [ensemble_bose, ensemble_fermi],
    "the three-route rendezvous: the contours of §7.2 meet this notebook's ensemble distributions",
    rtol=1e-6,
)
validate.check(
    qubit_vs_nF < 1e-14 and ladder_vs_nB < 1e-10,
    "and Movement I's single systems are the same functions identically (qubit → n_F, ladder → n_B)",
    f"|Δ| = {qubit_vs_nF:.1e}, {ladder_vs_nB:.1e}",
)

## Exercise 6 — The Maxwell–Boltzmann limit, and its ordering

Both statistics forget themselves at small fugacity — but not symmetrically. Cite
{eq}`eq-mb-limit`.

1. Show both $n_B$ and $n_F$ reduce to $ze^{-\beta\varepsilon}$ as $z \to 0$, and verify at
   $z = 0.01$ (relative deviations $\sim n_{\text{MB}}$, of order $z$).
2. Keep the next order — $n_B = n_{\text{MB}}(1+n_{\text{MB}})$, $n_F = n_{\text{MB}}
   (1-n_{\text{MB}})$ — and confirm the ordering $n_B > n_{\text{MB}} > n_F$ numerically across
   a range of $\varepsilon$.
3. Interpret (prose): over- and under-occupation relative to classical are bunching and
   exclusion showing through even in the dilute limit.
4. Defer explicitly: *where* the classical description is valid — the criterion
   $n\lambda^3 \ll 1$ and the thermal wavelength — is the next notebook's ([§7.8](classical-limit-thermal-wavelength.ipynb)) entire business.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.check(
    ordering_holds,
    "the common classical limit and its telltale ordering: n_B > MB > n_F at every energy",
)
validate.close(
    [float(dev_B[0]), float(dev_F[0])],
    [float(n_MB[0] / (1.0 - n_MB[0])), float(n_MB[0])],
    "the dilute corrections are ±n_MB to next order: bunching and exclusion in embryo",
    rtol=2e-2,
)

## Exercise 7 — Ensemble equivalence, by exact enumeration

The grand ensemble was a convenience; fixed-$N$ physics must agree — and we can watch it start
to. Cite {eq}`eq-bfd-equivalence`.

1. Use `canonical_occupations` (fermions at fixed $N$, every configuration enumerated via
   `itertools.combinations`) on a fixed energy band of $M$ evenly spaced levels,
   $\varepsilon_i = (i+\tfrac12)\,W/M$ with $W = 4$, at $\beta = 1$ — so that doubling $M$
   doubles the density of states rather than piling empty levels on top.
2. For each $(M, N) = (4, 2), (8, 4), (16, 8)$, match the grand ensemble's $\mu$ to
   $\langle N\rangle = N$ with `scipy.optimize.brentq` (the `mu_from_N` helper) and compare
   occupations mode by mode.
3. Confirm the maximum deviation shrinks roughly as $1/M$ as the system doubles, and plot the
   trend.
4. Connect (prose) to the classical ensemble-equivalence discussion of [§5.9](../05-classical-stat-mech/grand-canonical-ensemble-equivalence.ipynb): small systems care which
   ensemble describes them; thermodynamic matter does not — now demonstrated with Pauli in the
   game.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    deviations[0] > deviations[1] > deviations[2]
    and 0.3 < r21 < 0.7
    and 0.3 < r32 < 0.7,
    "ensemble equivalence, quantum edition: the canonical–grand deviation shrinks ~1/M",
    f"deviations {deviations[0]:.3f} → {deviations[1]:.3f} → {deviations[2]:.3f}",
)

## Exercise 8 — Bunching and antibunching

One variance formula, one sign, and the noise floor of the quantum world. Cite
{eq}`eq-bfd-fluctuations`.

1. Derive $\langle\Delta n^2\rangle = \langle n\rangle(1 \pm \langle n\rangle)$ from
   $(z\,\partial_z)^2\ln\Xi_i$ for both statistics.
2. Verify both exactly against brute Fock enumeration (the Exercise 2 modes, re-enumerated with
   `grand_enumerate` so this solution stands alone; boson grid at $n_{\max} = 120$, since second
   moments need more tail headroom than means) — fermion variances equal $n(1-n)$ to eight
   digits, boson variances equal $n(1+n)$ with the near-ceiling mode at variance $\approx 20.6$.
3. Establish the fermionic cap: $n(1-n) \le \tfrac14$, saturated at half filling — Pauli
   suppressing occupation noise (antibunching).
4. Interpret and connect (prose): the bosonic $+$ is the thermal bunching of [§7.5](oscillator-at-temperature.ipynb) generalized (and the
   photon-statistics seed for the photon-gas notebook); the fermionic $-$ is the quiet of
   electron shot noise and the Hanbury Brown–Twiss antibunching of fermions (horizons, named).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    varF8,
    varF_formula,
    "fermion fluctuations: ⟨Δn²⟩ = n(1−n) verified against the whole Fock space",
    rtol=1e-8,
)
validate.close(
    varB8,
    varB_formula,
    "boson fluctuations: ⟨Δn²⟩ = n(1+n), the near-ceiling mode at variance ≈ 20.6",
    rtol=1e-6,
)
validate.check(
    cap_max <= 0.25 + 1e-12 and abs(var_at_half - 0.25) < 1e-15,
    "the Pauli cap: n(1−n) ≤ 1/4, saturated exactly at half filling — antibunching",
    f"max {cap_max:.6f}",
)

## Exercise 9 — The two functions, earned

The volume has been collecting these two functions the way one collects sightings of a rare
bird — in a contour integral that knew nothing of temperature's ensembles, in a qubit and an
oscillator that knew nothing of each other — and this notebook finally produced the field guide.
Release the particle number, and the many-body problem falls apart into modes; each mode is a
small system we had already solved; and the only structural difference between the halves of the
material world is whether a mode holds two states or a ladder. From that one difference: a step
function that will organize metals and dead stars, a divergence held back by a ceiling on the
chemical potential, and noise that bunches for light while it hushes for electrons.

It is worth pausing on how the miracle was purchased. We did not get the distributions by being
cleverer about the constraint $\sum n_i = N$; we got them by *refusing to impose it*, and
trusting a reservoir to enforce it on average — then verifying, by exact enumeration, that the
fixed-$N$ world converges to the same answers as the system grows. Statistical mechanics keeps
teaching the same lesson in new clothes: the right ensemble is the one in which the system's
parts stop talking to each other.

The next notebook ([§7.8](classical-limit-thermal-wavelength.ipynb)) asks the bookkeeping question that makes all of this quantitative —
*when does a gas notice any of it?* — and in answering it (the thermal de Broglie wavelength,
the degeneracy criterion $n\lambda^3$) will supply Volume V's last missing derivation, the $N!$.

## Notebook summary

Movement II opens with the volume's central derivation, held to account at every step.

- **Occupation numbers** {eq}`eq-occupation`: the right coordinates for identical particles —
  $n_i \in \{0,1\}$ (Pauli, from the antisymmetry of [§6.20](../06-quantum-mechanics/identical-particles.ipynb)) or $\{0,1,2,\dots\}$ — with second
  quantization named as the many-body volume's horizon.
- **Factorization** {eq}`eq-grand-quantized`: releasing $N$ turns $\Xi$ into $\prod_i\Xi_i$ —
  verified against the *entire Fock space* to ten digits for both statistics, with the boson
  grid's truncation residual stated (the $r = 0.80$ mode) rather than hidden. Fixed $N$ couples
  every mode; the grand ensemble of [§5.9](../05-classical-stat-mech/grand-canonical-ensemble-equivalence.ipynb) was built for exactly this moment.
- **The two distributions** {eq}`eq-fermi-dirac`, {eq}`eq-bose-einstein`: $n_F$ from two Fock
  states (the $T\to0$ step with $n_F(\mu) = \tfrac12$ pinned; $\mu(0) = \varepsilon_F$
  previewed), $n_B$ from a geometric series whose convergence condition $\mu < \varepsilon_
  {\min}$ is the notebook's glowing fuse — the ceiling whose saturation will be Bose–Einstein
  condensation. Movement I's accidents closed as theorems: qubit $=$ fermion mode, ladder $=$
  boson mode (Planck at $\mu = 0$, checked by rung sums, not by the same formula).
- **The rendezvous** {eq}`eq-rendezvous`: contours ([§7.2](complex-analysis-applications.ipynb)), single systems ([§7.4](thermal-density-matrix.ipynb)/[§7.5](oscillator-at-temperature.ipynb)), and the
  ensemble land on one pair of functions — seven digits on the Matsubara forms, rounding level
  on the identifications. Independent machinery agreeing is the volume's methodological moral.
- **The MB limit** {eq}`eq-mb-limit`: both statistics $\to ze^{-\beta\varepsilon}$, with the
  dilute corrections $\pm n_{\text{MB}}$ enforcing $n_B > \text{MB} > n_F$ always — bunching and
  exclusion in embryo; the *validity* criterion deferred to [§7.8](classical-limit-thermal-wavelength.ipynb).
- **Ensemble equivalence** {eq}`eq-bfd-equivalence`: exact `combinations` enumeration vs the matched
  grand ensemble — deviations halving as $(M, N)$ double, the $1/M$ law with Pauli in the game.
- **Fluctuations** {eq}`eq-bfd-fluctuations`: $\langle\Delta n^2\rangle = n(1 \pm n)$, verified by
  enumeration to eight digits — bosonic bunching explosive near the ceiling (variance $20.6$ on
  the softest mode), fermionic antibunching capped at $\tfrac14$ — one sign separating laser
  light from electron shot noise.

Two functions, three routes, one field guide — and a fuse left lit for the movements ahead.

## Outlook

- **The classical limit made quantitative ([§7.8](classical-limit-thermal-wavelength.ipynb)).** The thermal de Broglie wavelength, the
  degeneracy criterion $n\lambda^3 \ll 1$, and the Gibbs $1/N!$ derived at last — Volume V's
  last unexplained factor.
- **The Fermi step put to work.** The ideal Fermi gas at $T = 0$, degeneracy pressure, and
  electrons in metals — the fermion movement.
- **The ceiling saturates.** Bose–Einstein condensation (the BEC notebook); $\mu = 0$ light and
  Planck's law (the photon-gas notebook).
- **Second quantization**, occupation numbers promoted to operators: the many-body volume's
  opening, named as a horizon.
- **Cross-reference** [§5.9](../05-classical-stat-mech/grand-canonical-ensemble-equivalence.ipynb) (the grand ensemble, put to work), [§6.20](../06-quantum-mechanics/identical-particles.ipynb) (the occupation restrictions), [§7.2](complex-analysis-applications.ipynb)
  (the contour route), [§7.4](thermal-density-matrix.ipynb)/[§7.5](oscillator-at-temperature.ipynb) (the single-system routes), [§7.3](statistical-toolkit.ipynb) (the integrals awaiting these
  occupations).

In [ ]:
from ecp.style import footer

footer()